## Polyvore Outfit Compatibility Training

* This trains a compatibility head on top of FashionCLIP embeddings (also used in fashion_clip_model.py and indofashion_service.py).

* Given two garment embeddings, the polyvore compatibility head predicts how compatible they are (either 0 or 1).





In [ ]:
# setup
!pip install -q transformers torch torchvision scikit-learn tqdm pillow

In [17]:
# imports
import os
import json
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [18]:
# for using hugging face dataset directly
!pip install -q datasets

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [24]:
from datasets import load_dataset

ds = load_dataset(
    "mvasil/polyvore-outfits",
    data_dir="data/disjoint"
)

print(ds)

README.md:   0%|          | 0.00/6.87k [00:00<?, ?B/s]

DatasetNotFoundError: Dataset 'mvasil/polyvore-outfits' is a gated dataset on the Hub. You must be authenticated to access it.

In [ ]:
print(ds["train"].column_names)

['item_id', 'image']


In [ ]:
print(ds["train"][0])

{'item_id': '193587212', 'image': {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\t\t\x08\n\x0c\x14\r\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c $.\' ",#\x1c\x1c(7),01444\x1f\'9=82<.342\xff\xdb\x00C\x01\t\t\t\x0c\x0b\x0c\x18\r\r\x182!\x1c!22222222222222222222222222222222222222222222222222\xff\xc0\x00\x11\x08\x01,\x01,\x03\x01"\x00\x02\x11\x01\x03\x11\x01\xff\xc4\x00\x1f\x00\x00\x01\x05\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\xff\xc4\x00\xb5\x10\x00\x02\x01\x03\x03\x02\x04\x03\x05\x05\x04\x04\x00\x00\x01}\x01\x02\x03\x00\x04\x11\x05\x12!1A\x06\x13Qa\x07"q\x142\x81\x91\xa1\x08#B\xb1\xc1\x15R\xd1\xf0$3br\x82\t\n\x16\x17\x18\x19\x1a%&\'()*456789:CDEFGHIJSTUVWXYZcdefghijstuvwxyz\x83\x84\x85\x86\x87\x88\x89\x8a\x92\x93\x94\x95\x96\x97\x98\x99\x9a\xa2\xa3\xa4\xa5\xa6\xa7\xa8\xa9\xaa\xb2\xb3\xb4\xb5\xb6\xb7\xb8\xb9\xba\xc2\xc

In [19]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# # config

# from pathlib import Path


# # Polyvore metadata/compatibility files
# DATA_ROOT = Path("/content/drive/MyDrive/MyStyla/polyvore_outfits")
# SPLIT_DIR = DATA_ROOT / "disjoint"

# # Hugging Face dataset containing the actual images


# EMBED_CACHE_PATH = Path("/content/drive/MyDrive/MyStyla/polyvore_embeddings.pt")
# HEAD_OUT_PATH = Path("/content/drive/MyDrive/MyStyla/polyvore_head.pth")
# RESULTS_PLOT_PATH = Path("/content/drive/MyDrive/MyStyla/polyvore_fitb_results.png")

# assert SPLIT_DIR.exists(), f"Disjoint folder not found: {SPLIT_DIR}"

# print("Split directory:", SPLIT_DIR)
# print("Hugging Face dataset:", ds)

Split directory: /content/drive/MyDrive/MyStyla/polyvore_outfits/disjoint


NameError: name 'ds' is not defined

In [20]:
from pathlib import Path

# Polyvore metadata/compatibility files
DATA_ROOT = Path("/content/drive/MyDrive/MyStyla/polyvore_outfits")
SPLIT_DIR = DATA_ROOT / "disjoint"

# Saved FashionCLIP embeddings
EMBED_CACHE_PATH = Path(
    "/content/drive/MyDrive/MyStyla/polyvore_embeddings.pt"
)

# Step 4 output
HEAD_OUT_PATH = Path(
    "/content/drive/MyDrive/MyStyla/polyvore_head.pth"
)

RESULTS_PLOT_PATH = Path(
    "/content/drive/MyDrive/MyStyla/polyvore_fitb_results.png"
)

assert SPLIT_DIR.exists(), f"Disjoint folder not found: {SPLIT_DIR}"
assert EMBED_CACHE_PATH.exists(), f"Embedding cache not found: {EMBED_CACHE_PATH}"

print("Split directory:", SPLIT_DIR)
print("Embedding cache:", EMBED_CACHE_PATH)

Split directory: /content/drive/MyDrive/MyStyla/polyvore_outfits/disjoint
Embedding cache: /content/drive/MyDrive/MyStyla/polyvore_embeddings.pt


## STEP 1: Load Outfit and Item Metadata

In [26]:
def load_outfit_json(path):
    """set_id -> {index: item_id}"""
    with open(path) as f:
        outfits = json.load(f)
    mapping = {}
    for outfit in outfits:
        set_id = str(outfit["set_id"])
        mapping[set_id] = {
            str(item["index"]): str(item["item_id"]) for item in outfit["items"]
        }
    return mapping


def load_compatibility_file(txt_path, outfit_map):
    """Returns list of (item_id_list, label) using the compatibility_*.txt split."""
    samples = []
    with open(txt_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            label = int(parts[0])
            item_ids = []
            for token in parts[1:]:
                set_id, index = token.split("_")
                item_id = outfit_map.get(set_id, {}).get(index)
                if item_id:
                    item_ids.append(item_id)
            if len(item_ids) >= 2:
                samples.append((item_ids, label))
    return samples


train_outfit_map = load_outfit_json(SPLIT_DIR / "train.json")
valid_outfit_map = load_outfit_json(SPLIT_DIR / "valid.json")
test_outfit_map = load_outfit_json(SPLIT_DIR / "test.json")

train_samples = load_compatibility_file(SPLIT_DIR / "compatibility_train.txt", train_outfit_map)
valid_samples = load_compatibility_file(SPLIT_DIR / "compatibility_valid.txt", valid_outfit_map)
test_samples = load_compatibility_file(SPLIT_DIR / "compatibility_test.txt", test_outfit_map)

print(f"train outfits: {len(train_samples)}, valid: {len(valid_samples)}, test: {len(test_samples)}")


train outfits: 33990, valid: 6000, test: 30290


## STEP 2: Extract FashionCLIP Embedding

### 2.1 Load FashionCLIP

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch
from tqdm import tqdm

CLIP_MODEL_NAME = "patrickjohncyh/fashion-clip"

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)

clip_model.eval()

print("FashionCLIP loaded.")
print("Device:", DEVICE)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

FashionCLIP loaded.
Device: cuda


### 2.2 Collect all item IDs we actually need

In [22]:
def all_item_ids(*sample_lists):
    ids = set()

    for samples in sample_lists:
        for item_ids, _ in samples:
            ids.update(item_ids)

    return sorted(ids)


needed_ids = all_item_ids(
    train_samples,
    valid_samples,
    test_samples
)

print("Number of unique item IDs needed:", len(needed_ids))

Number of unique item IDs needed: 152785


### 2.3 Create the embedding extraction function

In [ ]:
@torch.no_grad()
def extract_embeddings(item_ids, batch_size=32):
    embeddings = {}

    for i in tqdm(
        range(0, len(item_ids), batch_size),
        desc="Extracting FashionCLIP embeddings"
    ):
        batch_ids = item_ids[i:i + batch_size]

        images = []
        valid_ids = []

        for item_id in batch_ids:
            if item_id not in image_lookup:
                continue

            image = image_lookup[item_id]

            # Hugging Face Dataset images can come as:
            # 1. PIL Image
            # 2. {"bytes": ..., "path": ...}
            if isinstance(image, dict):
                if image.get("bytes") is not None:
                    import io
                    image = Image.open(
                        io.BytesIO(image["bytes"])
                    ).convert("RGB")
                elif image.get("path") is not None:
                    image = Image.open(
                        image["path"]
                    ).convert("RGB")
                else:
                    continue

            elif isinstance(image, Image.Image):
                image = image.convert("RGB")

            else:
                continue

            images.append(image)
            valid_ids.append(item_id)

        if not images:
            continue

        inputs = clip_processor(
            images=images,
            return_tensors="pt",
            padding=True
        )

        inputs = {
            k: v.to(DEVICE)
            for k, v in inputs.items()
        }

        # Get image representations
        output = clip_model.vision_model(**{
            "pixel_values": inputs["pixel_values"]
        })

        # IMPORTANT:
        # output is BaseModelOutputWithPooling,
        # so we take .pooler_output before doing math.
        pooled = output.pooler_output

        # Project vision features into CLIP embedding space
        features = clip_model.visual_projection(pooled)

        # L2 normalization
        features = features / features.norm(
            dim=-1,
            keepdim=True
        )

        for item_id, feat in zip(valid_ids, features.cpu()):
            embeddings[item_id] = feat

    return embeddings

### 2.4 Set the cache location

In [ ]:
EMBED_CACHE_PATH = Path(
    "/content/drive/MyDrive/MyStyla/polyvore_embeddings.pt"
)

print("Embedding cache:", EMBED_CACHE_PATH)

Embedding cache: /content/drive/MyDrive/MyStyla/polyvore_embeddings.pt


### 2.5 Extract or load the embeddings

In [23]:
# Recreate the image lookup from the Hugging Face dataset

train_hf = ds["train"]
valid_hf = ds["validation"]
test_hf = ds["test"]

image_lookup = {}

for split in [train_hf, valid_hf, test_hf]:
    for row in tqdm(split, desc="Building image lookup"):
        image_lookup[str(row["item_id"])] = row["image"]

print("Total images in lookup:", len(image_lookup))

NameError: name 'ds' is not defined

In [ ]:
# verify image lookup
example_id = needed_ids[0]

print("Example ID:", example_id)
print("Image found:", example_id in image_lookup)

if example_id in image_lookup:
    print("Image type:", type(image_lookup[example_id]))

Example ID: 100005237
Image found: True
Image type: <class 'dict'>


In [25]:
if EMBED_CACHE_PATH.exists():

    embeddings = torch.load(
        EMBED_CACHE_PATH,
        map_location="cpu"
    )

    print(f"Loaded {len(embeddings)} cached embeddings.")

else:

    print(f"Extracting embeddings for {len(needed_ids)} items...")

    embeddings = extract_embeddings(
        needed_ids,
        batch_size=32
    )

    torch.save(
        embeddings,
        EMBED_CACHE_PATH
    )

    print(f"Extracted and saved {len(embeddings)} embeddings.")

Loaded 152785 cached embeddings.


### Check if embedding extraction worked

In [ ]:
print("Number of embeddings:", len(embeddings))

first_id = next(iter(embeddings))

print("Example item ID:", first_id)
print("Embedding shape:", embeddings[first_id].shape)

Number of embeddings: 152785
Example item ID: 100005237
Embedding shape: torch.Size([512])


### STEP 3: Build Pairwise Dataset

Convert outfit-level samples into pairwise examples.

    Each sample is:
        (item_ids, label)

    where:
        item_ids = list of item IDs in an outfit
        label = 1 for compatible outfit
                0 for incompatible outfit

    Returns:
        X = tensor containing pairs of embeddings
        y = tensor containing compatibility labels

In [13]:
from itertools import combinations


def build_pairs(samples, embeddings):

    pair_a = []
    pair_b = []
    labels = []

    skipped_items = 0

    for item_ids, label in tqdm(
        samples,
        desc="Building pairs"
    ):

        # Keep only items that have FashionCLIP embeddings
        available_items = [
            item_id
            for item_id in item_ids
            if item_id in embeddings
        ]

        if len(available_items) < 2:
            skipped_items += 1
            continue

        # Create every unique pair inside the outfit
        for item_a, item_b in combinations(
            available_items, 2
        ):

            pair_a.append(embeddings[item_a])
            pair_b.append(embeddings[item_b])
            labels.append(float(label))

    X_a = torch.stack(pair_a)
    X_b = torch.stack(pair_b)

    y = torch.tensor(
        labels,
        dtype=torch.float32
    )

    print(f"Created {len(y):,} pairs")
    print(f"Skipped {skipped_items:,} outfits")
    print(f"Item A shape: {X_a.shape}")
    print(f"Item B shape: {X_b.shape}")
    print(f"Label shape: {y.shape}")

    return X_a, X_b, y

In [14]:
X_a_train, X_b_train, y_train = build_pairs(
    train_samples,
    embeddings
)

X_a_valid, X_b_valid, y_valid = build_pairs(
    valid_samples,
    embeddings
)

X_a_test, X_b_test, y_test = build_pairs(
    test_samples,
    embeddings
)

Building pairs: 100%|██████████| 33990/33990 [00:00<00:00, 45024.68it/s]


Created 385,784 pairs
Skipped 0 outfits
Item A shape: torch.Size([385784, 512])
Item B shape: torch.Size([385784, 512])
Label shape: torch.Size([385784])


Building pairs: 100%|██████████| 6000/6000 [00:00<00:00, 55318.50it/s]


Created 69,234 pairs
Skipped 0 outfits
Item A shape: torch.Size([69234, 512])
Item B shape: torch.Size([69234, 512])
Label shape: torch.Size([69234])


Building pairs: 100%|██████████| 30290/30290 [00:00<00:00, 139863.54it/s]


Created 330,946 pairs
Skipped 0 outfits
Item A shape: torch.Size([330946, 512])
Item B shape: torch.Size([330946, 512])
Label shape: torch.Size([330946])


### STEP 4: Model

* Small MLP on top of frozen FashionCLIP embeddings.
* FashionCLIP is not retrained. Its 1024-dimensional image embeddings are already extracted.
* For every pair of fashion items, the model receives:
  * embedding_a —representation of item A
  * embedding_b — representation of item B
  * |embedding_a - embedding_b| — absolute difference between the two embeddings

* Output: sigmoid probability the pair is compatible
* Output represents the probability that the two items are compatible.

In [27]:
class PolyvoreCompatibilityHead(nn.Module):

    def __init__(self, embed_dim=512, hidden_dim=256):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(embed_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, vec_a, vec_b):

        diff = torch.abs(vec_a - vec_b)

        x = torch.cat(
            [vec_a, vec_b, diff],
            dim=-1
        )

        return self.net(x).squeeze(-1)


model = PolyvoreCompatibilityHead(
    embed_dim=512
).to(DEVICE)

print(model)

PolyvoreCompatibilityHead(
  (net): Sequential(
    (0): Linear(in_features=1536, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [28]:
model = PolyvoreCompatibilityHead().to(DEVICE)

print(model)

PolyvoreCompatibilityHead(
  (net): Sequential(
    (0): Linear(in_features=1536, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=1, bias=True)
  )
)


### STEP 5: TRAIN


In [9]:
# create a dataset
class PairDataset(Dataset):

    def __init__(self, X_a, X_b, y):
        self.X_a = X_a
        self.X_b = X_b
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_a[idx],
            self.X_b[idx],
            self.y[idx]
        )

In [10]:
train_dataset = PairDataset(
    X_a_train,
    X_b_train,
    y_train
)

valid_dataset = PairDataset(
    X_a_valid,
    X_b_valid,
    y_valid
)

test_dataset = PairDataset(
    X_a_test,
    X_b_test,
    y_test
)

NameError: name 'X_a_train' is not defined

In [11]:
# data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=256,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

NameError: name 'train_dataset' is not defined

In [18]:
# check loader
vec_a, vec_b, label = next(iter(train_loader))

print("A:", vec_a.shape)
print("B:", vec_b.shape)
print("Label:", label.shape)

A: torch.Size([256, 512])
B: torch.Size([256, 512])
Label: torch.Size([256])


In [23]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

criterion = nn.BCEWithLogitsLoss()

EPOCHS = 15
best_val_auc = 0.0

for epoch in range(1, EPOCHS + 1):

    #train model
    model.train()

    total_loss = 0.0

    for vec_a, vec_b, label in train_loader:

        vec_a = vec_a.to(DEVICE)
        vec_b = vec_b.to(DEVICE)
        label = label.to(DEVICE)

        optimizer.zero_grad()

        logits = model(vec_a, vec_b)

        loss = criterion(
            logits,
            label
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * len(label)

    train_loss = total_loss / len(train_dataset)


    #validate model
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():

        for vec_a, vec_b, label in valid_loader:

            vec_a = vec_a.to(DEVICE)
            vec_b = vec_b.to(DEVICE)

            logits = model(
                vec_a,
                vec_b
            )

            probs = torch.sigmoid(
                logits
            ).cpu()

            all_probs.extend(
                probs.tolist()
            )

            all_labels.extend(
                label.tolist()
            )

    val_auc = roc_auc_score(
        all_labels,
        all_probs
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val AUC: {val_auc:.4f}"
    )


    # save best model
    if val_auc > best_val_auc:

        best_val_auc = val_auc

        torch.save(
            model.state_dict(),
            HEAD_OUT_PATH
        )

        print(
            f" Best model saved "
            f"(AUC = {val_auc:.4f})"
        )


print(
    f"\nBest validation AUC: "
    f"{best_val_auc:.4f}"
)

print(
    f"Model saved to: "
    f"{HEAD_OUT_PATH}"
)

Epoch 01 | Train Loss: 0.6425 | Val AUC: 0.7138
 Best model saved (AUC = 0.7138)
Epoch 02 | Train Loss: 0.6164 | Val AUC: 0.7323
 Best model saved (AUC = 0.7323)
Epoch 03 | Train Loss: 0.6021 | Val AUC: 0.7437
 Best model saved (AUC = 0.7437)
Epoch 04 | Train Loss: 0.5932 | Val AUC: 0.7479
 Best model saved (AUC = 0.7479)
Epoch 05 | Train Loss: 0.5856 | Val AUC: 0.7520
 Best model saved (AUC = 0.7520)
Epoch 06 | Train Loss: 0.5790 | Val AUC: 0.7557
 Best model saved (AUC = 0.7557)
Epoch 07 | Train Loss: 0.5738 | Val AUC: 0.7561
 Best model saved (AUC = 0.7561)
Epoch 08 | Train Loss: 0.5690 | Val AUC: 0.7577
 Best model saved (AUC = 0.7577)
Epoch 09 | Train Loss: 0.5639 | Val AUC: 0.7580
 Best model saved (AUC = 0.7580)
Epoch 10 | Train Loss: 0.5594 | Val AUC: 0.7556
Epoch 11 | Train Loss: 0.5548 | Val AUC: 0.7563
Epoch 12 | Train Loss: 0.5505 | Val AUC: 0.7572
Epoch 13 | Train Loss: 0.5473 | Val AUC: 0.7558
Epoch 14 | Train Loss: 0.5439 | Val AUC: 0.7561
Epoch 15 | Train Loss: 0.5411 |

### Evaluation: pairwise AUC + FITB(fill in the blank) accuracy

In [29]:
# load trained model
model.load_state_dict(
    torch.load(
        HEAD_OUT_PATH,
        map_location=DEVICE
    )
)

model.eval()

print("Trained model loaded.")

Trained model loaded.


In [30]:
FITB_PATH = SPLIT_DIR / "fill_in_blank_test.json"

print("FITB:", FITB_PATH)
print("Exists:", FITB_PATH.exists())

FITB: /content/drive/MyDrive/MyStyla/polyvore_outfits/disjoint/fill_in_blank_test.json
Exists: False


In [33]:
@torch.no_grad()
def score_outfit_compat(item_ids, embeddings):

    vecs = [
        embeddings[i]
        for i in item_ids
        if i in embeddings
    ]

    if len(vecs) < 2:
        return 0.0

    scores = []

    for a in range(len(vecs)):
        for b in range(a + 1, len(vecs)):

            va = vecs[a].unsqueeze(0).to(DEVICE)
            vb = vecs[b].unsqueeze(0).to(DEVICE)

            scores.append(
                torch.sigmoid(
                    model(va, vb)
                ).item()
            )

    return sum(scores) / len(scores)


FITB_PATH = SPLIT_DIR / "fill_in_the_blank_test.json"

print("FITB file:", FITB_PATH)
print("Exists:", FITB_PATH.exists())

with open(FITB_PATH) as f:
    fitb_questions = json.load(f)

print("Number of FITB questions:", len(fitb_questions))

FITB file: /content/drive/MyDrive/MyStyla/polyvore_outfits/disjoint/fill_in_the_blank_test.json
Exists: True
Number of FITB questions: 15145


In [35]:
#check fitb file
print(fitb_questions[0])

{'question': ['222049137_1', '222049137_2', '222049137_3', '222049137_4', '222049137_5'], 'blank_position': 6, 'answers': ['136139735_5', '171518178_4', '191247707_5', '222049137_6']}


In [36]:
# find relationship between data
q = fitb_questions[0]

print("Question:", q["question"])
print("Blank position:", q["blank_position"])
print("Answers:", q["answers"])

print("\nCorrect answer candidate should correspond to:")
print(f"{q['question'][0].split('_')[0]}_{q['blank_position']}")

Question: ['222049137_1', '222049137_2', '222049137_3', '222049137_4', '222049137_5']
Blank position: 6
Answers: ['136139735_5', '171518178_4', '191247707_5', '222049137_6']

Correct answer candidate should correspond to:
222049137_6


In [37]:
# fitb evaluator

@torch.no_grad()
def score_outfit_compat(item_ids, embeddings):

    vecs = [
        embeddings[item_id]
        for item_id in item_ids
        if item_id in embeddings
    ]

    if len(vecs) < 2:
        return 0.0

    scores = []

    for a in range(len(vecs)):
        for b in range(a + 1, len(vecs)):

            va = vecs[a].unsqueeze(0).to(DEVICE)
            vb = vecs[b].unsqueeze(0).to(DEVICE)

            probability = torch.sigmoid(
                model(va, vb)
            ).item()

            scores.append(probability)

    return sum(scores) / len(scores)


correct = 0
evaluated = 0
skipped = 0

for q in tqdm(
    fitb_questions,
    desc="Evaluating FITB"
):

    question_tokens = q["question"]
    answer_choices = q["answers"]
    blank_position = q["blank_position"]

    # Build the known outfit items
    question_items = []

    for token in question_tokens:

        set_id, index = token.split("_")

        item_id = test_outfit_map.get(
            set_id, {}
        ).get(index)

        if item_id is not None:
            question_items.append(item_id)

    if len(question_items) == 0:
        skipped += 1
        continue

    # Determine the official correct answer
    # from the original outfit and blank position.
    original_set_id = question_tokens[0].split("_")[0]

    correct_token = (
        f"{original_set_id}_{blank_position}"
    )

    if correct_token not in answer_choices:
        skipped += 1
        continue

    correct_idx = answer_choices.index(
        correct_token
    )

    best_choice = None
    best_score = -1.0

    # Score every candidate
    for choice_idx, token in enumerate(answer_choices):

        set_id, index = token.split("_")

        candidate_id = test_outfit_map.get(
            set_id, {}
        ).get(index)

        if (
            candidate_id is None
            or candidate_id not in embeddings
        ):
            continue

        candidate_outfit = (
            question_items + [candidate_id]
        )

        score = score_outfit_compat(
            candidate_outfit,
            embeddings
        )

        if score > best_score:
            best_score = score
            best_choice = choice_idx

    if best_choice is None:
        skipped += 1
        continue

    evaluated += 1

    if best_choice == correct_idx:
        correct += 1


fitb_accuracy = correct / evaluated

print()
print("=" * 50)
print(f"FITB Accuracy: {fitb_accuracy:.4f}")
print(f"Correct: {correct}")
print(f"Evaluated: {evaluated}")
print(f"Skipped: {skipped}")
print("=" * 50)

Evaluating FITB: 100%|██████████| 15145/15145 [04:03<00:00, 62.32it/s]


FITB Accuracy: 0.6291
Correct: 9528
Evaluated: 15145
Skipped: 0


### ROC curve plot

In [38]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(all_labels, all_probs)

plt.figure(figsize=(6, 5))
plt.plot(
    fpr,
    tpr,
    label=f"AUC = {test_auc:.3f}"
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(
    f"Polyvore Compatibility ROC "
    f"(FITB acc = {fitb_accuracy:.3f})"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    RESULTS_PLOT_PATH,
    dpi=150
)

plt.show()

print(
    f"Saved plot to {RESULTS_PLOT_PATH}"
)

ValueError: y_true takes value in {} and pos_label is not specified: either make y_true take value in {0, 1} or {-1, 1} or pass pos_label explicitly.